# Deep Learning 082 — Cross Attention

Companion notebook to the lesson. Cross attention is self-attention with **one line changed**:
the queries come from the decoder, the keys and values from the encoder. This notebook makes
that change, then checks the three things that follow from it — one of which is a hard
constraint that is rarely stated.

| Step | What we measure |
|---|---|
| the shapes | the weight matrix is **rectangular**, n_out × n_in |
| row sums | each output token spends **1.0** of attention on the source |
| the constraint | the output lies in the **convex hull** of the encoder values |
| is it Luong? | difference **0.00** — exactly, not approximately |
| does alignment emerge? | **~85%** of the mass on the right token vs **16.7%** chance |
| the cost | a decoder block is **1.33×** an encoder block |

Needs `torch` (CPU is fine). Part D trains for about a minute.

In [ ]:
import math
import torch
import torch.nn as nn

D, H = 64, 4

class CrossAttention(nn.Module):
    '''Q from the decoder; K and V from the encoder.  That is the entire change.'''
    def __init__(self, d=D, h=H):
        super().__init__()
        self.h, self.dk = h, d // h
        self.wq, self.wk = nn.Linear(d, d), nn.Linear(d, d)
        self.wv, self.wo = nn.Linear(d, d), nn.Linear(d, d)

    def forward(self, dec, enc):                      # <- TWO sequences, not one
        n_out, n_in = len(dec), len(enc)
        q = self.wq(dec).view(n_out, self.h, self.dk).transpose(0, 1)
        k = self.wk(enc).view(n_in,  self.h, self.dk).transpose(0, 1)
        v = self.wv(enc).view(n_in,  self.h, self.dk).transpose(0, 1)
        w = torch.softmax(q @ k.transpose(-2, -1) / math.sqrt(self.dk), dim=-1)
        out = (w @ v).transpose(0, 1).reshape(n_out, self.h * self.dk)
        return self.wo(out), w

Compare that `forward` with lesson 077's multi-head attention. The only difference is the
signature: two arguments instead of one, and `wk`/`wv` applied to `enc` while `wq` is applied
to `dec`.

## Part A — What the one change does to the shapes

In [ ]:
torch.manual_seed(1082)
enc = torch.randn(5, D)         # "I like eating ice cream" -> 5 encoder vectors
dec = torch.randn(3, D)         # 3 Hindi words generated so far

xa = CrossAttention()
out, w = xa(dec, enc)

print(f"encoder output (K, V) : {tuple(enc.shape)}")
print(f"decoder input  (Q)    : {tuple(dec.shape)}")
print(f"attention weights     : {tuple(w.shape[-2:])}   <- RECTANGULAR")
print(f"output                : {tuple(out.shape)}   <- one per OUTPUT token")
print(f"\nrow sums (head 0)     : {w[0].sum(-1)}")
assert w.shape[-2:] == (3, 5)

Self-attention's matrix is square because a sequence is compared with itself. Here two
different sequences meet, so **the output length is free** — a 5-word English sentence can
become a 3-word Hindi one.

Read the row sums and the block explains itself: each output token is handed **exactly 1.0**
of attention and must spend all of it across the **source** sentence. The output sequence is
literally expressed in terms of the input one, which is what a translation is.

## Part B — A constraint that follows, and that nobody states

Before `W_o`, each output row is $\sum_j w_{ij} v_j$ with $w_{ij} \ge 0$ summing to 1. That is
not just a weighted sum, it is a **convex combination** — so the output is trapped inside the
convex hull of the encoder's value vectors. It sounds abstract. It is checkable.

In [ ]:
torch.manual_seed(2082)
enc, dec = torch.randn(7, D), torch.randn(4, D)
xa1 = CrossAttention(h=1)                      # one head, so the algebra is visible

with torch.no_grad():
    v = xa1.wv(enc)
    _, w = xa1(dec, enc)
    mixed = w[0] @ v                           # the output BEFORE W_o

basis, _ = torch.linalg.qr(v.T)                # orthonormal basis of span(V)
resid = mixed - mixed @ basis @ basis.T
print(f"distance from span of the encoder values : {resid.norm():.2e}")
print(f"coordinates below the encoder minimum    : {(mixed < v.min(0).values - 1e-5).sum()}")
print(f"coordinates above the encoder maximum    : {(mixed > v.max(0).values + 1e-5).sum()}")
print(f"||mixed|| / mean ||v||                   : "
      f"{(mixed.norm(dim=1).mean() / v.norm(dim=1).mean()):.4f}")

**The decoder can redistribute, but it cannot invent.** At this block it is strictly a mixer
of encoder content: if the source sentence never encoded a fact, no weighting of the encoder's
outputs can recover it. That is the architectural reason an encoder-decoder cannot add
information that was not in the source — worth holding on to when reasoning about what such
models can and cannot hallucinate.

The last line is the other half: averaging **shrinks**, to about 0.41 of the length of the
vectors it mixed. That is the same shrinkage lesson 080 measured as token collapse, and it is
why `W_o` and the feed-forward network follow.

## Part C — Is it really Bahdanau/Luong attention?

Lessons 067 and 068 built attention for the RNN encoder-decoder: score the decoder state
against each encoder hidden state, softmax, take the weighted sum. Textbooks say cross
attention is "the same idea". Test whether it is stronger than that.

In [ ]:
torch.manual_seed(3082)
h_enc, s_dec = torch.randn(6, D), torch.randn(4, D)

# Luong's dot-product attention, written directly
alpha = torch.softmax(s_dec @ h_enc.T, dim=-1)
c_luong = alpha @ h_enc

# cross attention: one head, identity projections, no 1/sqrt(d_k)
xa1 = CrossAttention(h=1)
with torch.no_grad():
    for lin in (xa1.wq, xa1.wk, xa1.wv, xa1.wo):
        lin.weight.copy_(torch.eye(D)); lin.bias.zero_()
    w = torch.softmax(xa1.wq(s_dec) @ xa1.wk(h_enc).T, dim=-1)
    c_cross = xa1.wo(w @ xa1.wv(h_enc))

print(f"max |alpha_luong - w_cross| = {(alpha - w).abs().max():.2e}")
print(f"max |c_luong - c_cross|     = {(c_luong - c_cross).abs().max():.2e}")

Not `1e-7` — **zero**, because it is the same sequence of floating-point operations.

So cross attention adds three things to 2015: **learned projections** (similarity measured in
a space the model chooses rather than raw hidden-state space), the **$1/\sqrt{d_k}$** scaling
from lesson 074, and **multiple heads**. The idea is unchanged. What the transformer removed
was the RNN underneath it, not the attention.

## Part D — Does the alignment actually emerge?

The heat maps in translation papers show a model quietly discovering which source word each
target word corresponds to, with nothing supervising it. That is testable if the correct
answer is known in advance — so: the target is the source **reversed**, meaning output
position $i$ must align to input position $n-1-i$.

In [ ]:
torch.manual_seed(4082)
V, N, d = 20, 6, D

emb_in = nn.Embedding(V, d)
pos_in = nn.Parameter(torch.randn(N, d) * 0.1)
pos_out = nn.Parameter(torch.randn(N, d) * 0.1)
xa = CrossAttention(d=d, h=1)
head = nn.Linear(d, V)

params = list(emb_in.parameters()) + [pos_in, pos_out] + \
         list(xa.parameters()) + list(head.parameters())
opt = torch.optim.Adam(params, lr=3e-3)

def batch(bs=64):
    src = torch.randint(0, V, (bs, N))
    return src, src.flip(1)

for step in range(1, 1201):
    src, tgt = batch()
    loss = 0.0
    for b in range(len(src)):
        out, _ = xa(pos_out.expand(N, d), emb_in(src[b]) + pos_in)
        loss = loss + nn.functional.cross_entropy(head(out), tgt[b])
    loss = loss / len(src)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 300 == 0:
        print(f"step {step:>4}  loss {loss.item():.4f}")

In [ ]:
with torch.no_grad():
    src, tgt = batch(200)
    want = torch.arange(N - 1, -1, -1)
    correct = mass = peak = 0.0
    acc = torch.zeros(N, N)
    for b in range(len(src)):
        out, w = xa(pos_out.expand(N, d), emb_in(src[b]) + pos_in)
        A = w[0]
        acc += A
        correct += (head(out).argmax(-1) == tgt[b]).float().mean().item()
        mass += A[torch.arange(N), want].mean().item()
        peak += (A.argmax(-1) == want).float().mean().item()
    n = len(src); acc /= n

print(f"token accuracy                        : {100*correct/n:.1f}%")
print(f"attention mass on the CORRECT token   : {mass/n:.4f}")
print(f"chance, if the row were uniform       : {1/N:.4f}")
print(f"rows peaking at the correct token     : {100*peak/n:.1f}%\n")

print("      " + "".join(f"in{j:<5}" for j in range(N)))
for i in range(N):
    print(f"out{i}  " + "".join(f"{acc[i, j]:<7.3f}" for j in range(N)))

**The anti-diagonal is the alignment, and nothing put it there.** The only training signal was
"produce these output tokens"; the correspondence between positions was inferred. This is what
the heat maps in the translation literature are showing, and it is the clearest evidence that
cross attention does the job it was designed for rather than merely occupying space in the
diagram.

## Part E — What it costs

In [ ]:
d, d_ff = 512, 2048
attn = 4 * (d*d + d)
ff = (d*d_ff + d_ff) + (d_ff*d + d)
ln = 2*d

dec_block = 2*attn + ff + 3*ln          # masked self, cross, feed-forward
enc_block = attn + ff + 2*ln

print(f"one attention block (any flavour) : {attn:>10,}")
print(f"one DECODER block                 : {dec_block:>10,}")
print(f"one ENCODER block                 : {enc_block:>10,}")
print(f"ratio                             : {dec_block/enc_block:>10.2f}x\n")

stacks = 6*(dec_block + enc_block)
embed = 37_000 * d
print(f"both stacks                       : {stacks:>10,}")
print(f"the embedding table               : {embed:>10,}")
print(f"stacks / vocabulary               : {stacks/embed:>10.2f}x")

Cross attention costs exactly what any attention block costs — the shapes are identical, only
the source of $K$ and $V$ differs — so a decoder block is **1.33×** an encoder block.

Lesson 080 found the six-block encoder level with the embedding table at 0.16%. Adding the
decoder puts the stacks at **2.33×** the vocabulary. Six architectures in this course had the
vocabulary-facing layer dominating the parameter count; **this is where that finishes.**

## What to take away

- **Cross attention is self-attention with $Q$ from the decoder and $K$, $V$ from the
  encoder.** The paper calls it encoder-decoder attention.
- **The matrix is rectangular**, $n_{out} \times n_{in}$, with rows summing to 1 over the
  source. The output sequence is expressed in terms of the input one.
- **The output is a convex combination of the encoder's values** — inside their span and their
  range. The decoder redistributes; it cannot invent.
- **It is exactly Luong attention** when reduced to one head with identity projections:
  difference 0.00.
- **Alignment emerges unsupervised** — about 85% of the mass on the correct token against
  16.7% chance. (The lesson's run of `scripts/gen_cross_attention.py` reports 86.3%; the exact
  figure moves a little with initialisation, the anti-diagonal does not.)

**Exercises**

1. In Part D, give the queries real content (embed the target prefix) instead of positions
   only. Does the alignment get sharper or blurrier, and why?
2. Raise the head count in Part D from 1 to 4 and average the heads. Do all heads learn the
   same alignment?
3. Change the task from "reverse" to "shift by two". Does the learned matrix move to the
   right diagonal?
4. Verify the convex-hull constraint from Part B *after* `W_o`. Does it still hold? What does
   that tell you about what `W_o` is for?